In [1]:
import pandas as pd 
import numpy as np
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv( "data/rewriting_pairs.csv" )
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (11830, 2)

Columns:
['toxic', 'neutral']


## Get Source Target

In [3]:
source_texts = df["toxic"].astype(str).values
target_texts = df["neutral"].astype(str).values

## Adding Special Tokens to target sentences

In [4]:
target_texts = ["<start> " + text + " <end>" for text in target_texts]

In [5]:
source_train, source_temp, target_train, target_temp = train_test_split(
    source_texts,
    target_texts,
    test_size=0.20,
    random_state=42
)

source_val, source_test, target_val, target_test = train_test_split(
    source_temp,
    target_temp,
    test_size=0.50,
    random_state=42
)

print("\nDataset split:") 
print("Training:", len(source_train))
print("Validation:", len(source_val))
print("Testing:", len(source_test))


Dataset split:
Training: 9464
Validation: 1183
Testing: 1183


## create source tokenizer

In [6]:
source_vocab_size = 20000

source_tokeinizer = Tokenizer(
    num_words=source_vocab_size,
    oov_token="<OOV>"
)

source_tokeinizer.fit_on_texts(source_train)

## create target tokenizer

In [7]:
target_vocab_size = 20000

target_tokenizer = Tokenizer(
    num_words = target_vocab_size,
    oov_token="<OOV>",
    filters=""
)

target_tokenizer.fit_on_texts(target_train)

print("\nSpecial tokens:")
print("START:", target_tokenizer.word_index.get("<start>"))
print("END:", target_tokenizer.word_index.get("<end>"))

print("\nFirst 20 target vocabulary entries:")
print(
    list(target_tokenizer.word_index.items())[:20]
)


Special tokens:
START: 2
END: 3

First 20 target vocabulary entries:
[('<OOV>', 1), ('<start>', 2), ('<end>', 3), ('the', 4), ('i', 5), ('you', 6), ('to', 7), ('is', 8), ('a', 9), ('and', 10), ('that', 11), ('of', 12), ('this', 13), ('.', 14), ('it', 15), ('not', 16), ('are', 17), ('in', 18), (',', 19), ('for', 20)]


## convert source text to sequences

In [8]:
encoder_train = source_tokeinizer.texts_to_sequences( source_train )
encoder_val = source_tokeinizer.texts_to_sequences( source_val )
encoder_test = source_tokeinizer.texts_to_sequences( source_test )

In [9]:
decoder_train = target_tokenizer.texts_to_sequences( target_train )
decoder_val = target_tokenizer.texts_to_sequences( target_val )
decoder_test = target_tokenizer.texts_to_sequences( target_test )

## determine max sequence len

In [10]:
source_lengths = [
    len(sequence)
    for sequence in encoder_train
]

target_lengths = [
    len(sequence)
    for sequence in decoder_train
]

In [11]:
max_source_length = int( np.percentile(source_lengths, 95) )
max_target_length = int( np.percentile(target_lengths, 95) )

In [12]:
max_source_length = max( max_source_length, 10 )
max_target_length = max( max_target_length, 10 )

print("\nSequence lengths:")
print("Maximum source length:", max_source_length)
print("Maximum target length:", max_target_length)


Sequence lengths:
Maximum source length: 17
Maximum target length: 19


In [13]:
encoder_train = pad_sequences(
    encoder_train,
    maxlen = max_source_length,
    padding = "post",
    truncating = "post"
)

encoder_val = pad_sequences(
    encoder_val,
    maxlen = max_source_length,
    padding = "post",
    truncating = "post"
)

encoder_test = pad_sequences(
    encoder_test,
    maxlen = max_source_length,
    padding = "post",
    truncating = "post"
)

In [14]:
decoder_train = pad_sequences(
    decoder_train,
    maxlen=max_target_length,
    padding="post",
    truncating="post"
)

decoder_val = pad_sequences(
    decoder_val,
    maxlen=max_target_length,
    padding="post",
    truncating="post"
)

decoder_test = pad_sequences(
    decoder_test,
    maxlen=max_target_length,
    padding="post",
    truncating="post"
)

In [15]:
decoder_input_train = decoder_train[:, :-1]
decoder_target_train = decoder_train[:, 1:]

decoder_input_val = decoder_val[:, :-1]
decoder_target_val = decoder_val[:, 1:]

decoder_input_test = decoder_test[:, :-1]
decoder_target_test = decoder_test[:, 1:]

In [16]:
actual_source_vocab_size = min(
    source_vocab_size,
    len(source_tokeinizer.word_index) + 1
)

actual_target_vocab_size = min(
    target_vocab_size,
    len(target_tokenizer.word_index) + 1
)

print("\nVocabulary sizes:")
print( "Source vocabulary:", actual_source_vocab_size )
print( "Target vocabulary:", actual_target_vocab_size )



Vocabulary sizes:
Source vocabulary: 10320
Target vocabulary: 11706


In [17]:
print("\nEncoder shapes:")
print("Train:", encoder_train.shape) 
print("Validation:", encoder_val.shape)
print("Test:", encoder_test.shape)

print("\nDecoder input shapes:")
print("Train:", decoder_input_train.shape) 
print("Validation:", decoder_input_val.shape)
print("Test:", decoder_input_test.shape)

print("\ndecoder target shapes:")
print("Train:", decoder_target_train.shape) 
print("Validation:", decoder_target_val.shape)
print("Test:", decoder_target_test.shape)


Encoder shapes:
Train: (9464, 17)
Validation: (1183, 17)
Test: (1183, 17)

Decoder input shapes:
Train: (9464, 18)
Validation: (1183, 18)
Test: (1183, 18)

decoder target shapes:
Train: (9464, 18)
Validation: (1183, 18)
Test: (1183, 18)


In [18]:
with open( "data/source_tokenizer.pkl", "wb" ) as f: pickle.dump( source_tokeinizer, f )
with open( "data/target_tokenizer.pkl", "wb" ) as f: pickle.dump( target_tokenizer, f )

In [19]:
with open( "data/rewriting_train.pkl", "wb" ) as f: pickle.dump( ( encoder_train, decoder_input_train, decoder_target_train ), f )

In [20]:
with open( "data/rewriting_val.pkl", "wb" ) as f: pickle.dump( ( encoder_val, decoder_input_val, decoder_target_val ), f )

In [21]:
with open( "data/rewriting_test.pkl", "wb" ) as f: pickle.dump( ( encoder_test, decoder_input_test, decoder_target_test ), f )

In [22]:
sequence_info = { "source_vocab_size": actual_source_vocab_size, "target_vocab_size": actual_target_vocab_size, "max_source_length": max_source_length, "max_target_length": max_target_length }

In [23]:
with open( "data/rewriting_sequence_info.pkl", "wb" ) as f: pickle.dump( sequence_info, f )